In [2]:
import torch
import torch.nn as nn

torch.manual_seed(0)

In [5]:
# A model with Dropout — randomly zeros 50% of neurons during training
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Dropout(p=0.5),   # 50% of neurons randomly silenced during training
    nn.Linear(8, 1)
)

In [7]:
x = torch.rand(1,4) # one sample

In [23]:
# In train mode — Dropout is active, output changes every call
model.train()
print(model(x))   # tensor([[0.1473]], grad_fn=...)
print(model(x))   # tensor([[0.3021]], grad_fn=...)  ← different!
print(model(x))   # tensor([[0.0891]], grad_fn=...)  ← different again!

tensor([[0.4208]], grad_fn=<AddmmBackward0>)
tensor([[0.5690]], grad_fn=<AddmmBackward0>)
tensor([[0.1175]], grad_fn=<AddmmBackward0>)


In [25]:
# In eval mode — Dropout is off, output is consistent
model.eval()
with torch.no_grad():
    print(model(x))   # tensor([[0.2204]])
    print(model(x))   # tensor([[0.2204]])  ← same ✅
    print(model(x))   # tensor([[0.2204]])  ← same ✅

tensor([[0.2520]])
tensor([[0.2520]])
tensor([[0.2520]])


In [ ]:
def evaluate(model, X, y, loss_fn):
    """Evaluate model — always call this, never evaluate in train mode."""

    model.eval()   # flip to eval mode

    with torch.no_grad():
        logits = model(X)                           # forward pass, no grad
        loss   = loss_fn(logits, y)                 # compute loss

        # For classification: convert logits to predicted class
        predicted = torch.argmax(logits, dim=1)     # index of highest score
        correct   = (predicted == y).sum().item()   # count correct
        accuracy  = correct / len(y) * 100          # as percentage

    return loss.item(), accuracy

# Usage
loss, acc = evaluate(model, X_val, y_val, loss_fn)
print(f"Val Loss: {loss:.4f} | Val Accuracy: {acc:.1f}%")